# Why was Emily St. John Mandel split into two authors?

On the live site her checkouts were divided across two records — `emily st john mandel, 1979` (38,363) and `Mandel, Emily St. John` (13,629) — even though they're the same person. 
This notebook traces exactly where that split comes from, using the committed `author_map.json` and the clustering code in this folder.

**The punchline up front:** the split is real and lived in the map, but `normalize()` is *not* the cause — it keys every spelling identically. The map was just incomplete relative to it. Run this notebook from the `data-prep/` folder.

In [1]:
import json
from cluster_authors import normalize, name_variant_remap

author_map = json.load(open("author_map.json"))   # raw catalog name -> canonical name
print(f"{len(author_map):,} name mappings in the committed map")

449,968 name mappings in the committed map


## 1. The committed map only covers *some* of her spellings

The catalog spells her several ways. Which ones are in the map, and where do they point?

In [2]:
spellings = [
    "Mandel, Emily St. John",
    "Mandel, Emily St. John, 1979-",
    "Emily St. John Mandel",
    "Emily St. John Mandel, 1979-",
]
for s in spellings:
    if s in author_map:
        print(f"{s!r:34} -> {author_map[s]!r}   (in map)")
    else:
        print(f"{s!r:34} -> (NOT in map: stays its own author)")

'Mandel, Emily St. John'           -> (NOT in map: stays its own author)
'Mandel, Emily St. John, 1979-'    -> 'emily st john mandel, 1979'   (in map)
'Emily St. John Mandel'            -> 'emily st john mandel, 1979'   (in map)
'Emily St. John Mandel, 1979-'     -> (NOT in map: stays its own author)


Two of the four spellings — including the plain `Mandel, Emily St. John` — **aren't in the map at all.** 
The build uses `COALESCE(map, raw)`: any name not in the map is kept as-is, so each of those becomes its **own** author record.

## 2. So the build produces *three* separate "Mandel" authors

In [3]:
canonical = {s: author_map.get(s, s) for s in spellings}
for s, c in canonical.items():
    print(f"{s!r:34} -> {c!r}")
print()
print("distinct author records the build produces:")
for c in sorted(set(canonical.values())):
    print("   ", repr(c))

'Mandel, Emily St. John'           -> 'Mandel, Emily St. John'
'Mandel, Emily St. John, 1979-'    -> 'emily st john mandel, 1979'
'Emily St. John Mandel'            -> 'emily st john mandel, 1979'
'Emily St. John Mandel, 1979-'     -> 'Emily St. John Mandel, 1979-'

distinct author records the build produces:
    'Emily St. John Mandel, 1979-'
    'Mandel, Emily St. John'
    'emily st john mandel, 1979'


Three records for one person — her ~52k checkouts scattered across them. **That's the duplication you saw on the site.**

## 3. Is `normalize()` to blame? No — and this is the important part

`normalize()` builds the match key used to decide two names are the same person. If it gave these spellings *different* keys, that would explain the split. Let's check:

In [4]:
for s in spellings:
    print(f"{s!r:34} -> normalize() = {normalize(s)!r}")

'Mandel, Emily St. John'           -> normalize() = 'emily st john mandel'
'Mandel, Emily St. John, 1979-'    -> normalize() = 'emily st john mandel'
'Emily St. John Mandel'            -> normalize() = 'emily st john mandel'
'Emily St. John Mandel, 1979-'     -> normalize() = 'emily st john mandel'


**Every spelling normalizes to the same key, `emily st john mandel`.** So `normalize()` is working correctly.

The problem is that the *committed map* doesn't contain all of these raw spellings. `cluster_authors.py` builds the map from an aggregate of catalog records (cached / processed incrementally), so a spelling that wasn't in that aggregate when the map was last generated never gets a mapping — even though `normalize()` *would* group it. 
In other words: **the map is stale/incomplete relative to the current `normalize()`.** Regenerating the map would fix these — but you'd have to remember to, every time.

## 4. The build-time backstop fixes it regardless

`name_variant_remap()` (called from `build_data.py` on every build) re-merges the split *canonical* records by their bag-of-words, so it doesn't depend on the map being fresh:

In [5]:
# the two main records the build produced, with their checkout totals:
split_records = [("Mandel, Emily St. John", 13629), ("emily st john mandel, 1979", 38363)]

remap = name_variant_remap(split_records)
print("merge decision (loser -> winner):", remap)
print()
winner = (set(n for n, _ in split_records) - set(remap)).pop()
print(f"-> both fold into the most-borrowed form {winner!r}; her checkouts reunite (~{13629+38363:,})")

merge decision (loser -> winner): {'Mandel, Emily St. John': 'emily st john mandel, 1979'}

-> both fold into the most-borrowed form 'emily st john mandel, 1979'; her checkouts reunite (~51,992)


## Takeaway

- The split was **real** and lived in the committed `author_map.json`, which only mapped *some* of her spellings.
- `normalize()` itself is **correct** — it keys every spelling identically. The map was just incomplete relative to it.
- The build-time `name_variant_remap()` merges the split records by bag-of-words on **every build**, so the fix holds even when the committed map lags behind the code.
- Longer term: regenerating `author_map.json` (or making the map builder include every raw spelling `normalize()` recognizes) removes the split at the source; the backstop then just confirms it.

## Appendix — the spellings in the raw open data

Proof the multiple spellings really exist in the source (needs network):

In [6]:
import urllib.request, urllib.parse
TOKEN = "bHWsGtRFRP9x8Hl8lYivqM1hQ"
BOOK = ("((usageclass='Physical' AND materialtype='BOOK') OR "
        "(usageclass='Digital' AND upper(materialtype) IN ('EBOOK','AUDIOBOOK')))")
soql = ("SELECT creator, sum(checkouts) AS c "
        f"WHERE upper(creator) like '%MANDEL%' AND upper(creator) like '%EMILY%' AND {BOOK} "
        "GROUP BY creator ORDER BY c DESC")
url = "https://data.seattle.gov/resource/tmmm-ytt6.json?" + urllib.parse.urlencode({"$query": soql, "$$app_token": TOKEN})
try:
    for r in json.loads(urllib.request.urlopen(url, timeout=60).read()):
        print(f"{int(float(r['c'])):>7,}  {r['creator']!r}  ->  build canonical: {author_map.get(r['creator'], r['creator'])!r}")
except Exception as e:
    print("(network needed to run this cell):", e)

 38,363  'Emily St. John Mandel'  ->  build canonical: 'emily st john mandel, 1979'
 13,630  'Mandel, Emily St. John, 1979-'  ->  build canonical: 'emily st john mandel, 1979'
